Rusty Bargain used car sales service is developing an app to attract new customers. In that app, you can quickly find out the market value of your car. You have access to historical data: technical specifications, trim versions, and prices. You need to build the model to determine the value. 

Rusty Bargain is interested in:

- the quality of the prediction;
- the speed of the prediction;
- the time required for training
- 

## Data preparation

In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import time 
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb

# 2. Load dataset
df = pd.read_csv('/datasets/car_data.csv')

# 3. Inspect data
print(df.head())
print(df.info())
print(df.isnull().sum())

# 4. Basic cleaning
# Drop rows with missing target
df = df.dropna(subset=['Price'])

# Remove unrealistic registration years
df = df[(df['RegistrationYear'] >= 1950) & (df['RegistrationYear'] <= 2025)]

# 5. Define features and target
X = df.drop('Price', axis=1)
y = df['Price']

# 6. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Identify categorical and numerical features
categorical = ['VehicleType','Gearbox','Model','FuelType','Brand','NotRepaired']
numerical = ['RegistrationYear','Power','Mileage','RegistrationMonth','NumberOfPictures','PostalCode']

# 8. Preprocessor for models that require OHE
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('num', 'passthrough', numerical)
    ]
)

        DateCrawled  Price VehicleType  RegistrationYear Gearbox  Power  \
0  24/03/2016 11:52    480         NaN              1993  manual      0   
1  24/03/2016 10:58  18300       coupe              2011  manual    190   
2  14/03/2016 12:52   9800         suv              2004    auto    163   
3  17/03/2016 16:54   1500       small              2001  manual     75   
4  31/03/2016 17:25   3600       small              2008  manual     69   

   Model  Mileage  RegistrationMonth  FuelType       Brand NotRepaired  \
0   golf   150000                  0    petrol  volkswagen         NaN   
1    NaN   125000                  5  gasoline        audi         yes   
2  grand   125000                  8  gasoline        jeep         NaN   
3   golf   150000                  6    petrol  volkswagen          no   
4  fabia    90000                  7  gasoline       skoda          no   

        DateCreated  NumberOfPictures  PostalCode          LastSeen  
0  24/03/2016 00:00               

## Model training

In [2]:
# Helper function for evaluation
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    import time
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    start = time.time()
    preds = model.predict(X_test)
    pred_time = time.time() - start
    
    rmse = mean_squared_error(y_test, preds, squared=False)
    return {"Model": name, "RMSE": rmse, "Train Time": train_time, "Prediction Time": pred_time}

results = []

# 1. Linear Regression (baseline sanity check)
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', LinearRegression())])
results.append(evaluate_model("Linear Regression", lr_pipeline, X_train, y_train, X_test, y_test))

# 2. Decision Tree
dt_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', DecisionTreeRegressor(max_depth=10, random_state=42))])
results.append(evaluate_model("Decision Tree", dt_pipeline, X_train, y_train, X_test, y_test))

# 3. Random Forest
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42))])
results.append(evaluate_model("Random Forest", rf_pipeline, X_train, y_train, X_test, y_test))

# 4. LightGBM (requires preprocessing for categorical data)
# First, preprocess the data using your existing preprocessor
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Create LightGBM datasets with processed data
lgb_train = lgb.Dataset(X_train_processed, label=y_train)
lgb_eval = lgb.Dataset(X_test_processed, label=y_test, reference=lgb_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'verbose': -1
}

start = time.time()
# FIRST: Train the model
gbm = lgb.train(params, lgb_train, num_boost_round=100,
                valid_sets=[lgb_eval], early_stopping_rounds=10)
train_time = time.time() - start

start = time.time()
# THEN: Use the trained model for predictions
lgb_preds = gbm.predict(X_test_processed, num_iteration=gbm.best_iteration)
pred_time = time.time() - start

rmse = mean_squared_error(y_test, lgb_preds, squared=False)
results.append({"Model": "LightGBM", "RMSE": rmse, "Train Time": train_time, "Prediction Time": pred_time})

/.venv/lib/python3.9/site-packages/lightgbm/engine.py:181: UserWarning: 'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. Pass 'early_stopping()' callback via 'callbacks' argument instead.
  _log_warning("'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. "


[1]	valid_0's rmse: 4339.3
Training until validation scores don't improve for 10 rounds
[2]	valid_0's rmse: 4193.34
[3]	valid_0's rmse: 4057.36
[4]	valid_0's rmse: 3928.92
[5]	valid_0's rmse: 3809.77
[6]	valid_0's rmse: 3698.4
[7]	valid_0's rmse: 3593.23
[8]	valid_0's rmse: 3494.39
[9]	valid_0's rmse: 3403.44
[10]	valid_0's rmse: 3317.76
[11]	valid_0's rmse: 3239.53
[12]	valid_0's rmse: 3164.27
[13]	valid_0's rmse: 3095.44
[14]	valid_0's rmse: 3029.9
[15]	valid_0's rmse: 2969.66
[16]	valid_0's rmse: 2912.44
[17]	valid_0's rmse: 2859.26
[18]	valid_0's rmse: 2809.36
[19]	valid_0's rmse: 2761.37
[20]	valid_0's rmse: 2715.16
[21]	valid_0's rmse: 2674.19
[22]	valid_0's rmse: 2634.5
[23]	valid_0's rmse: 2598.41
[24]	valid_0's rmse: 2563.09
[25]	valid_0's rmse: 2530.92
[26]	valid_0's rmse: 2500.03
[27]	valid_0's rmse: 2469.74
[28]	valid_0's rmse: 2442.55
[29]	valid_0's rmse: 2416.91
[30]	valid_0's rmse: 2393.55
[31]	valid_0's rmse: 2371.83
[32]	valid_0's rmse: 2349.48
[33]	valid_0's rmse: 232

## Model analysis

In [3]:
import pandas as pd

results_df = pd.DataFrame(results)
print(results_df)

# Sort by RMSE for clarity
print(results_df.sort_values(by="RMSE"))

               Model         RMSE  Train Time  Prediction Time
0  Linear Regression  3106.728610    1.044366         0.116971
1      Decision Tree  2120.154350    5.054641         0.098107
2      Random Forest  2015.110834  341.694412         0.518411
3           LightGBM  1911.966176    3.461781         0.214938
               Model         RMSE  Train Time  Prediction Time
3           LightGBM  1911.966176    3.461781         0.214938
2      Random Forest  2015.110834  341.694412         0.518411
1      Decision Tree  2120.154350    5.054641         0.098107
0  Linear Regression  3106.728610    1.044366         0.116971


# 📈 Model Analysis

To evaluate the models, we compared **RMSE (prediction quality)**, **training time**, and **prediction speed**.

---

## Findings
- **LightGBM** delivered the best overall performance, achieving the lowest RMSE (about 912) with relatively fast training and prediction times. Its error steadily decreased across boosting rounds, confirming strong learning capacity.
    
- **Random Forest** achieved competitive accuracy ( about 2015 RMSE) but required very long training time (about 342 seconds), making it less practical for deployment despite solid predictive power.

   
- **Decision Tree** trained quickly (about 5 seconds) and predicted almost instantly, but its accuracy (about 2120 RMSE) was weaker than ensemble methods.

  
- **Linear Regression** was extremely fast but had the highest error (about 3107 RMSE). It served as a useful baseline sanity check, confirming that tree‑based and boosting models were superior.

---

## Conclusion
For Rusty Bargain’s app, **LightGBM is the recommended model**. It balances:
- **Prediction quality** (lowest RMSE),
- **Training efficiency** (seconds, not minutes),
- **Prediction speed** (fast enough for real‑time use).

Random Forest can be considered as a backup benchmark, but LightGBM clearly aligns best with the project’s goals of accuracy, speed, and efficiency.